In [27]:
import biom
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

In [19]:
pca = PCA(n_components=100, whiten=True)

In [2]:
biom_table = biom.load_table("../../data/table_gut_all.biom")

In [25]:
model = RandomForestClassifier(random_state=11, n_jobs=1, n_estimators=200)

In [28]:
out_dir = Path("Data/disease_data/results_with_rf_PCA")
out_dir.mkdir(parents=True, exist_ok=True)

### IBD

In [42]:
metadata = pd.read_csv("Data/disease_data/IBD/metadata.tsv", sep="\t", index_col=0)
metadata.head(n=3)

,study,diagnosis,group,age,sex,bmi,country,region,pcr_primers,platform,disease_name,title,disease_name_ab,site,subject_id
sample,,,,,,,,,,,,,,,
ERR1368879,RISK_PRISM_f,CD,1,19.0,male,NaN,515F-806R,V4,Illumina MiSeq,USA,NaN,NaN,IBD,feces,NaN
ERR1368880,RISK_PRISM_f,CD,1,26.0,male,NaN,515F-806R,V4,Illumina MiSeq,USA,NaN,NaN,IBD,feces,NaN
ERR1368881,RISK_PRISM_f,UC,1,55.0,male,NaN,515F-806R,V4,Illumina MiSeq,USA,NaN,NaN,IBD,feces,NaN


In [43]:
table_1 = biom.load_table("Data/disease_data/IBD/PRJNA324147/train_loo.biom")
table_2 = biom.load_table("Data/disease_data/IBD/PRJNA324147/test_loo.biom")
table_df = table_1.merge(table_2)
table_df = table_df.norm(axis="sample")
table_df = table_df.to_dataframe().T

In [44]:
table_df_pca = pca.fit_transform(table_df.values)
table_df_pca = pd.DataFrame(data=table_df_pca, index=table_df.index.values)

In [45]:
metadata = metadata.loc[table_df_pca.index.values]

In [46]:
study_list = metadata.study.unique()

In [47]:
for s in study_list:
    out_dir = Path(f"Data/disease_data/results_with_rf_PCA/IBD_{s}")
    out_dir.mkdir(parents=True, exist_ok=True)
    train_sid = metadata.loc[[i != s for i in metadata.study.values]].index.values
    test_sid = metadata.loc[[i == s for i in metadata.study.values]].index.values

    Xtr = table_df_pca.loc[train_sid].values
    Xte = table_df_pca.loc[test_sid].values

    ytr = (metadata.loc[train_sid, 'group'] == 1).astype(int).values
    yte = (metadata.loc[test_sid, 'group'] == 1).astype(int).values

    model.fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    proba = model.predict_proba(Xte)[:, 1]
    df = pd.DataFrame({
        "sample_id": test_sid,
        "true_label": yte,
        "prob": proba,
    })
    df.to_csv(f"{out_dir}/pred_prob.csv", index=False)
    print(f"study: {s}; auc: {auc}")

study: RISK_PRISM_f; auc: 0.517469545957918
study: qiita_1629; auc: 0.5374977135540516
study: qiita_2538; auc: 0.5632163323782235
study: PRJNA324147; auc: 0.47924528301886793
study: PRJNA368966; auc: 0.6383186705767351
study: PRJNA422193; auc: 0.7046736502820306
study: PRJNA431126; auc: 0.627607425949103
study: PRJNA450340; auc: 0.5963499731615672


### CRC

In [48]:
metadata = pd.read_csv("Data/disease_data/CRC/metadata.tsv", sep="\t", index_col=0)
table_1 = biom.load_table("Data/disease_data/CRC/PRJDB11845/train_loo.biom")
table_2 = biom.load_table("Data/disease_data/CRC/PRJDB11845/test_loo.biom")
table_df = table_1.merge(table_2)
table_df = table_df.norm(axis="sample")
table_df = table_df.to_dataframe().T

table_df_pca = pca.fit_transform(table_df.values)
table_df_pca = pd.DataFrame(data=table_df_pca, index=table_df.index.values)

In [49]:
table_df_pca = pca.fit_transform(table_df.values)
table_df_pca = pd.DataFrame(data=table_df_pca, index=table_df.index.values)

In [50]:
metadata = metadata.loc[table_df_pca.index.values]
study_list = metadata.study.unique()

In [51]:
for s in study_list:
    out_dir = Path(f"Data/disease_data/results_with_rf_PCA/CRC_{s}")
    out_dir.mkdir(parents=True, exist_ok=True)
    train_sid = metadata.loc[[i != s for i in metadata.study.values]].index.values
    test_sid = metadata.loc[[i == s for i in metadata.study.values]].index.values

    Xtr = table_df_pca.loc[train_sid].values
    Xte = table_df_pca.loc[test_sid].values

    ytr = (metadata.loc[train_sid, 'group'] == 1).astype(int).values
    yte = (metadata.loc[test_sid, 'group'] == 1).astype(int).values

    model.fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    proba = model.predict_proba(Xte)[:, 1]
    df = pd.DataFrame({
        "sample_id": test_sid,
        "true_label": yte,
        "prob": proba,
    })
    df.to_csv(f"{out_dir}/pred_prob.csv", index=False)
    print(f"study: {s}; auc: {auc}")

study: PRJDB11845; auc: 0.5038461538461538
study: PRJEB36789; auc: 0.7097270280661284
study: PRJEB6070; auc: 0.5856097560975609
study: PRJNA824020; auc: 0.48018648018648014
study: PRJNA290926; auc: 0.6333230769230769
study: PRJNA318004; auc: 0.6140502354788069
study: PRJNA430990; auc: 0.5824774354186119
